# RGB/MS Residual-scSE U-Net: seeds 123 and 2026

Both modalities; this is a Residual-scSE U-Net, not the plain U-Net. Final manuscript aggregate must be reconciled with the original result exports.

Source: `gd-second.ipynb`, zero-based cell 13.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
"""
GD-RGBMS-DB — RGB-ONLY + MS-ONLY MULTI-SEED REPRODUCIBILITY
=============================================================

Runs ONLY seeds 123 and 2026 for the two single-modality baselines:
    1) RGB-only: R, G, B
    2) MS-only : Green, Red-Edge, NIR

This uses the same Residual-scSE U-Net backbone/protocol associated with the
paper's seed-42 modality results:
    RGB-only seed 42: Dice 0.7602, IoU 0.6131, P 0.7464, R 0.7744, th 0.64
    MS-only  seed 42: Dice 0.7798, IoU 0.6391, P 0.7532, R 0.8085, th 0.40

Protocol retained:
- Official GD-RGBMS-DB split: 280 train / 60 validation / 60 test
- 896 x 672 input
- AdamW, lr=1e-4, wd=1e-4
- Batch 2, gradient accumulation 2 (effective 4)
- Max 70 epochs, early-stop patience 12
- ReduceLROnPlateau factor 0.5, patience 4, min lr 1e-6
- Loss = 0.35 BCE + 0.25 focal + 0.40 Dice
- GroupNorm + Residual + scSE backbone
- Horizontal/vertical flips for training
- RGB gain/bias only for RGB imagery; MS radiometry unchanged
- Validation-only checkpoint selection and threshold selection
- Fine threshold search 0.20..0.80 step 0.01
- Final test: no TTA
- num_workers=0 for Kaggle stability

No seed-42 model or checkpoint is required. Seed 42 is NOT rerun.
"""

from __future__ import annotations

import csv
import json
import random
import shutil
import time
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm import tqdm as _tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

BASE_DIR = Path("/kaggle/input/datasets/hemal7/gd-rgbms-db")

WORK_DIR = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path("/content")
)

MASTER_OUT_DIR = WORK_DIR / "gd_rgb_ms_residual_scse_seeds123_2026"
MASTER_OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_SEEDS = [123, 2026]
MODALITIES = ["RGB", "MS"]

TARGET_SIZE = (896, 672)  # PIL: width, height
BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
MAX_EPOCHS = 70
PATIENCE = 12
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

PREFERRED_GN_GROUPS = 8
ENCODER_CHANNELS = [32, 64, 128, 248]
EXPECTED_PARAMETER_COUNT_3CH = 7_901_827

CACHE_TRAIN_AND_VALIDATION = True
SHOW_PROGRESS = False

MASK_POSITIVE_RGB = np.array([240, 128, 128], dtype=np.int16)

ALL_CHANNEL_NAMES = [
    "R", "G", "B",
    "Green", "Red-Edge", "NIR",
]

# Training-split channel statistics already used in the controlled experiments.
ALL_MEAN = np.array(
    [
        0.41819795966148376,
        0.46573567390441895,
        0.2516770660877228,
        0.21822915971279144,
        0.16206900775432587,
        0.23585167527198792,
    ],
    dtype=np.float32,
)

ALL_STD = np.array(
    [
        0.1540451943874359,
        0.1569177359342575,
        0.13375557959079742,
        0.07147330790758133,
        0.05124884843826294,
        0.04640943184494972,
    ],
    dtype=np.float32,
)

TRAIN_THRESHOLDS = np.round(
    np.arange(0.20, 0.851, 0.05), 2
).tolist()

FINAL_THRESHOLDS = np.round(
    np.arange(0.20, 0.801, 0.01), 2
).tolist()

try:
    BILINEAR = Image.Resampling.BILINEAR
    NEAREST = Image.Resampling.NEAREST
except AttributeError:
    BILINEAR = Image.BILINEAR
    NEAREST = Image.NEAREST


def tqdm(*args, **kwargs):
    kwargs["disable"] = not SHOW_PROGRESS
    return _tqdm(*args, **kwargs)


# =============================================================================
# 2. REPRODUCIBILITY / DATASET DISCOVERY
# =============================================================================

def read_list(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def valid_root(path: Path):
    required = [
        path / "RGB",
        path / "MS",
        path / "Mark",
        path / "train_list.txt",
        path / "val_list.txt",
        path / "test_list.txt",
    ]

    if not all(p.exists() for p in required):
        return False

    try:
        return (
            len(read_list(path / "train_list.txt")) == 280
            and len(read_list(path / "val_list.txt")) == 60
            and len(read_list(path / "test_list.txt")) == 60
        )
    except Exception:
        return False


def find_dataset_root():
    if BASE_DIR.exists() and valid_root(BASE_DIR):
        return BASE_DIR

    roots = [
        Path("/kaggle/input"),
        Path("/content"),
        Path("/content/drive/MyDrive"),
    ]

    candidates = []

    for root in roots:
        if not root.exists():
            continue

        for train_file in root.rglob("train_list.txt"):
            candidate = train_file.parent
            if valid_root(candidate):
                candidates.append(candidate)

    if not candidates:
        raise FileNotFoundError(
            "Could not locate GD-RGBMS-DB with RGB/, MS/, Mark/ and "
            "the official 280/60/60 split files. Set BASE_DIR manually."
        )

    candidates.sort(
        key=lambda p: (
            "gd-rgbms" not in str(p).lower(),
            len(str(p)),
        )
    )

    return candidates[0]


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


DATASET_DIR = find_dataset_root()
RGB_DIR = DATASET_DIR / "RGB"
MS_DIR = DATASET_DIR / "MS"
MARK_DIR = DATASET_DIR / "Mark"

TRAIN_NAMES = read_list(DATASET_DIR / "train_list.txt")
VAL_NAMES = read_list(DATASET_DIR / "val_list.txt")
TEST_NAMES = read_list(DATASET_DIR / "test_list.txt")

assert len(TRAIN_NAMES) == 280
assert len(VAL_NAMES) == 60
assert len(TEST_NAMES) == 60
assert not (set(TRAIN_NAMES) & set(VAL_NAMES))
assert not (set(TRAIN_NAMES) & set(TEST_NAMES))
assert not (set(VAL_NAMES) & set(TEST_NAMES))

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
PIN_MEMORY = DEVICE.type == "cuda"

print("=" * 100)
print("GD-RGBMS-DB — RGB-ONLY + MS-ONLY | SEEDS 123 AND 2026")
print("=" * 100)
print("Dataset    :", DATASET_DIR)
print("Device     :", DEVICE)
print("Split      :", len(TRAIN_NAMES), len(VAL_NAMES), len(TEST_NAMES))
print("Input size :", TARGET_SIZE)
print("Backbone   : Residual-scSE U-Net")
print("Modalities :", MODALITIES)
print("Seeds      :", RUN_SEEDS)
print("Seed 42    : NOT rerun")
print("=" * 100)


# =============================================================================
# 3. RAW DATA / CACHE
# =============================================================================

def find_path(directory: Path, stem: str, extensions):
    for ext in extensions:
        p = directory / f"{stem}{ext}"
        if p.exists():
            return p

    raise FileNotFoundError(
        f"Could not find paired file for {stem} in {directory}"
    )


def make_binary_mask(image: Image.Image):
    arr = np.asarray(image.convert("RGB"), dtype=np.int16)
    distance = np.abs(arr - MASK_POSITIVE_RGB).sum(axis=2)
    return (distance < 80).astype(np.float32)


def load_raw(name: str):
    stem = Path(name).stem

    rgb_path = find_path(
        RGB_DIR,
        stem,
        [".JPG", ".jpg", ".JPEG", ".jpeg"],
    )

    ms_path = find_path(
        MS_DIR,
        stem,
        [".PNG", ".png", ".JPG", ".jpg"],
    )

    mark_path = find_path(
        MARK_DIR,
        stem,
        [".PNG", ".png", ".JPG", ".jpg"],
    )

    with Image.open(rgb_path) as item:
        rgb_image = item.convert("RGB").resize(
            TARGET_SIZE,
            BILINEAR,
        )

    with Image.open(ms_path) as item:
        ms_image = item.convert("RGB").resize(
            TARGET_SIZE,
            BILINEAR,
        )

    with Image.open(mark_path) as item:
        mask_image = item.convert("RGB").resize(
            TARGET_SIZE,
            NEAREST,
        )

    rgb = np.asarray(
        rgb_image,
        dtype=np.float32,
    ).copy() / 255.0

    ms = np.asarray(
        ms_image,
        dtype=np.float32,
    ).copy() / 255.0

    mask = make_binary_mask(mask_image)

    six_channel = np.concatenate(
        [rgb, ms],
        axis=2,
    ).astype(np.float32)

    return six_channel, mask, stem


RAM_CACHE = {}


def preload_cache(names):
    if not CACHE_TRAIN_AND_VALIDATION:
        return

    for name in tqdm(
        names,
        desc="Preloading train+validation cache",
    ):
        image, mask, stem = load_raw(name)
        RAM_CACHE[name] = (
            image.astype(np.float16),
            mask.astype(np.uint8),
            stem,
        )

    gb = sum(
        image.nbytes + mask.nbytes
        for image, mask, _ in RAM_CACHE.values()
    ) / (1024 ** 3)

    print(
        f"RAM cache: {len(RAM_CACHE)} samples "
        f"(~{gb:.2f} GB)"
    )


preload_cache(TRAIN_NAMES + VAL_NAMES)


# =============================================================================
# 4. MODALITY DATASET
# =============================================================================

def modality_info(modality: str):
    if modality == "RGB":
        return {
            "indices": slice(0, 3),
            "channels": ALL_CHANNEL_NAMES[:3],
            "mean": ALL_MEAN[:3],
            "std": ALL_STD[:3],
        }

    if modality == "MS":
        return {
            "indices": slice(3, 6),
            "channels": ALL_CHANNEL_NAMES[3:6],
            "mean": ALL_MEAN[3:6],
            "std": ALL_STD[3:6],
        }

    raise ValueError(modality)


class GDDataset(Dataset):
    def __init__(self, names, modality, augment=False):
        self.names = list(names)
        self.modality = modality
        self.augment = augment

        info = modality_info(modality)
        self.indices = info["indices"]
        self.mean = info["mean"].reshape(1, 1, 3)
        self.std = info["std"].reshape(1, 1, 3)

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]

        if name in RAM_CACHE:
            six_channel, mask, stem = RAM_CACHE[name]
            six_channel = six_channel.astype(np.float32)
            mask = mask.astype(np.float32)
        else:
            six_channel, mask, stem = load_raw(name)

        image = six_channel[..., self.indices].copy()

        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=1).copy()
                mask = np.flip(mask, axis=1).copy()

            if random.random() < 0.5:
                image = np.flip(image, axis=0).copy()
                mask = np.flip(mask, axis=0).copy()

            # Paper protocol: perturb RGB only; leave MS radiometry unchanged.
            if self.modality == "RGB" and random.random() < 0.75:
                gain = np.random.uniform(
                    0.88,
                    1.12,
                    size=(1, 1, 3),
                ).astype(np.float32)

                bias = np.random.uniform(
                    -0.035,
                    0.035,
                    size=(1, 1, 3),
                ).astype(np.float32)

                image = np.clip(
                    image * gain + bias,
                    0.0,
                    1.0,
                )

        image = (
            (image - self.mean) / self.std
        ).astype(np.float32)

        return (
            torch.from_numpy(
                image.transpose(2, 0, 1)
            ).float(),
            torch.from_numpy(mask[None]).float(),
            stem,
        )


def make_loaders(seed: int, modality: str):
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        GDDataset(
            TRAIN_NAMES,
            modality,
            augment=True,
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    val_loader = DataLoader(
        GDDataset(
            VAL_NAMES,
            modality,
            augment=False,
        ),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    test_loader = DataLoader(
        GDDataset(
            TEST_NAMES,
            modality,
            augment=False,
        ),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    return train_loader, val_loader, test_loader


# =============================================================================
# 5. RESIDUAL-scSE U-NET
# =============================================================================

def group_norm(channels: int):
    groups = min(PREFERRED_GN_GROUPS, channels)

    while channels % groups:
        groups -= 1

    return nn.GroupNorm(
        groups,
        channels,
        eps=1e-5,
        affine=True,
    )


class SCSE(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()

        hidden = max(channels // reduction, 1)

        self.channel = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )

        self.spatial = nn.Sequential(
            nn.Conv2d(channels, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return (
            x * self.channel(x)
            + x * self.spatial(x)
        )


class ResidualSCSEBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        dropout: float = 0.0,
    ):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                3,
                padding=1,
                bias=False,
            ),
            group_norm(out_channels),
            nn.SiLU(inplace=True),
            nn.Dropout2d(dropout)
            if dropout
            else nn.Identity(),
            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=1,
                bias=False,
            ),
            group_norm(out_channels),
        )

        self.skip = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    1,
                    bias=False,
                ),
                group_norm(out_channels),
            )
        )

        self.attention = SCSE(out_channels)

    def forward(self, x):
        residual = F.silu(
            self.block(x) + self.skip(x),
            inplace=True,
        )

        return self.attention(residual)


class UpBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
        dropout=0.0,
    ):
        super().__init__()

        self.conv = ResidualSCSEBlock(
            in_channels + skip_channels,
            out_channels,
            dropout,
        )

    def forward(self, x, skip):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        return self.conv(
            torch.cat([x, skip], dim=1)
        )


class ResidualSCSEUNet(nn.Module):
    def __init__(
        self,
        input_channels=3,
        channels=ENCODER_CHANNELS,
    ):
        super().__init__()

        c1, c2, c3, c4 = channels

        self.pool = nn.MaxPool2d(2)

        self.encoder1 = ResidualSCSEBlock(
            input_channels,
            c1,
        )
        self.encoder2 = ResidualSCSEBlock(
            c1,
            c2,
        )
        self.encoder3 = ResidualSCSEBlock(
            c2,
            c3,
            0.05,
        )
        self.encoder4 = ResidualSCSEBlock(
            c3,
            c4,
            0.10,
        )

        self.center = ResidualSCSEBlock(
            c4,
            c4 * 2,
            0.10,
        )

        self.up4 = UpBlock(
            c4 * 2,
            c4,
            c4,
            0.10,
        )
        self.up3 = UpBlock(
            c4,
            c3,
            c3,
            0.05,
        )
        self.up2 = UpBlock(
            c3,
            c2,
            c2,
        )
        self.up1 = UpBlock(
            c2,
            c1,
            c1,
        )

        self.output = nn.Conv2d(
            c1,
            1,
            1,
        )

    def forward(self, x):
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        e4 = self.encoder4(self.pool(e3))
        center = self.center(self.pool(e4))

        d4 = self.up4(center, e4)
        d3 = self.up3(d4, e3)
        d2 = self.up2(d3, e2)
        d1 = self.up1(d2, e1)

        return self.output(d1)


# =============================================================================
# 6. LOSS / METRICS
# =============================================================================

class ComboLoss(nn.Module):
    def forward(self, logits, targets):
        targets = targets.float()

        bce_map = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
        )

        bce = bce_map.mean()

        probabilities = torch.sigmoid(logits)

        pt = (
            probabilities * targets
            + (1.0 - probabilities) * (1.0 - targets)
        )

        focal = (
            bce_map * ((1.0 - pt) ** 2.0)
        ).mean()

        dims = tuple(range(1, logits.ndim))

        intersection = (
            probabilities * targets
        ).sum(dims)

        denominator = (
            probabilities.sum(dims)
            + targets.sum(dims)
        )

        dice_loss = 1.0 - (
            (2.0 * intersection + 1e-6)
            / (denominator + 1e-6)
        ).mean()

        return (
            0.35 * bce
            + 0.25 * focal
            + 0.40 * dice_loss
        )


def confusion_tensor(probabilities, masks, threshold):
    prediction = probabilities >= threshold
    target = masks >= 0.5

    return [
        int((prediction & target).sum().item()),
        int((prediction & ~target).sum().item()),
        int((~prediction & target).sum().item()),
        int((~prediction & ~target).sum().item()),
    ]


def confusion_numpy(probability, mask, threshold):
    prediction = probability >= threshold
    target = mask >= 0.5

    return [
        int(np.logical_and(prediction, target).sum()),
        int(np.logical_and(prediction, ~target).sum()),
        int(np.logical_and(~prediction, target).sum()),
        int(np.logical_and(~prediction, ~target).sum()),
    ]


def add_values(a, b):
    return [
        int(x + y)
        for x, y in zip(a, b)
    ]


def metrics(values):
    tp, fp, fn, tn = values
    eps = 1e-7

    return {
        "dice": (2 * tp + eps) / (
            2 * tp + fp + fn + eps
        ),
        "iou": (tp + eps) / (
            tp + fp + fn + eps
        ),
        "precision": (tp + eps) / (
            tp + fp + eps
        ),
        "recall": (tp + eps) / (
            tp + fn + eps
        ),
        "accuracy": (tp + tn + eps) / (
            tp + fp + fn + tn + eps
        ),
    }


def save_csv(path: Path, rows):
    if not rows:
        return

    with open(
        path,
        "w",
        newline="",
        encoding="utf-8",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=list(rows[0].keys()),
        )
        writer.writeheader()
        writer.writerows(rows)


def safe_load(path):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


# =============================================================================
# 7. ONE RUN
# =============================================================================

def run_experiment(seed: int, modality: str):
    seed_everything(seed)

    train_loader, val_loader, test_loader = make_loaders(
        seed,
        modality,
    )

    run_dir = (
        MASTER_OUT_DIR
        / f"seed_{seed}"
        / modality.lower()
    )
    run_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = (
        run_dir
        / f"best_{modality.lower()}_residual_scse_seed{seed}.pt"
    )

    model = ResidualSCSEUNet(
        input_channels=3,
    ).to(DEVICE)

    parameter_count = sum(
        p.numel()
        for p in model.parameters()
    )

    if parameter_count != EXPECTED_PARAMETER_COUNT_3CH:
        raise RuntimeError(
            f"Architecture mismatch: expected "
            f"{EXPECTED_PARAMETER_COUNT_3CH:,}, "
            f"got {parameter_count:,}."
        )

    criterion = ComboLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=DEVICE.type == "cuda",
    )

    info = modality_info(modality)

    print("\n" + "=" * 100)
    print(
        f"{modality}-ONLY | SEED {seed} | "
        "RESIDUAL-scSE U-NET"
    )
    print("=" * 100)
    print("Channels   :", info["channels"])
    print("Parameters :", f"{parameter_count:,}")

    def train_epoch(epoch):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0

        for step, (images, masks, _) in enumerate(
            tqdm(
                train_loader,
                desc=f"{modality} seed {seed} train {epoch}",
                leave=False,
            ),
            start=1,
        ):
            images = images.to(
                DEVICE,
                non_blocking=True,
            )
            masks = masks.to(
                DEVICE,
                non_blocking=True,
            )

            with torch.amp.autocast(
                "cuda",
                enabled=DEVICE.type == "cuda",
            ):
                logits = model(images)
                loss = criterion(logits, masks)
                scaled_loss = (
                    loss / ACCUMULATION_STEPS
                )

            scaler.scale(scaled_loss).backward()
            total_loss += loss.item()

            if (
                step % ACCUMULATION_STEPS == 0
                or step == len(train_loader)
            ):
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0,
                )

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        return total_loss / max(len(train_loader), 1)

    @torch.no_grad()
    def validate_checkpoint():
        model.eval()

        totals = {
            threshold: [0, 0, 0, 0]
            for threshold in TRAIN_THRESHOLDS
        }

        loss_total = 0.0

        for images, masks, _ in tqdm(
            val_loader,
            desc=f"{modality} seed {seed} validation",
            leave=False,
        ):
            images = images.to(
                DEVICE,
                non_blocking=True,
            )
            masks = masks.to(
                DEVICE,
                non_blocking=True,
            )

            with torch.amp.autocast(
                "cuda",
                enabled=DEVICE.type == "cuda",
            ):
                logits = model(images)
                loss_total += criterion(
                    logits,
                    masks,
                ).item()

            probabilities = torch.sigmoid(logits)

            for threshold in TRAIN_THRESHOLDS:
                totals[threshold] = add_values(
                    totals[threshold],
                    confusion_tensor(
                        probabilities,
                        masks,
                        threshold,
                    ),
                )

        rows = []

        for threshold, values in totals.items():
            row = metrics(values)
            row["threshold"] = float(threshold)
            row["loss"] = loss_total / max(len(val_loader), 1)
            rows.append(row)

        return max(
            rows,
            key=lambda item: item["dice"],
        )

    history = []
    best_val_dice = -1.0
    best_epoch = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        start = time.time()

        train_loss = train_epoch(epoch)
        validation = validate_checkpoint()

        scheduler.step(validation["dice"])

        current_lr = float(
            optimizer.param_groups[0]["lr"]
        )

        minutes = (
            time.time() - start
        ) / 60.0

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": validation["loss"],
            "val_dice": validation["dice"],
            "val_iou": validation["iou"],
            "val_precision": validation["precision"],
            "val_recall": validation["recall"],
            "val_accuracy": validation["accuracy"],
            "val_threshold": validation["threshold"],
            "learning_rate": current_lr,
            "minutes": minutes,
        }

        history.append(row)

        print(
            f"{modality} | seed {seed} | "
            f"epoch {epoch:03d}/{MAX_EPOCHS} | "
            f"loss {train_loss:.4f} | "
            f"val Dice {validation['dice']:.4f} | "
            f"IoU {validation['iou']:.4f} | "
            f"P {validation['precision']:.4f} | "
            f"R {validation['recall']:.4f} | "
            f"th {validation['threshold']:.2f} | "
            f"lr {current_lr:.2e}"
        )

        if validation["dice"] > best_val_dice:
            best_val_dice = float(
                validation["dice"]
            )
            best_epoch = epoch

            torch.save(
                {
                    "model": model.state_dict(),
                    "epoch": epoch,
                    "best_validation_dice": best_val_dice,
                    "coarse_validation_threshold": validation["threshold"],
                    "seed": seed,
                    "modality": modality,
                    "channels": info["channels"],
                    "architecture": "ResidualSCSEUNet",
                    "encoder_channels": ENCODER_CHANNELS,
                    "parameter_count": parameter_count,
                    "target_size": TARGET_SIZE,
                    "mean": info["mean"].tolist(),
                    "std": info["std"].tolist(),
                },
                checkpoint_path,
            )

        if epoch - best_epoch >= PATIENCE:
            print(
                f"{modality} seed {seed}: early stopping "
                f"after {PATIENCE} non-improving epochs."
            )
            break

    save_csv(
        run_dir / "training_history.csv",
        history,
    )

    # -------------------------------------------------------------------------
    # Reload best checkpoint
    # -------------------------------------------------------------------------

    checkpoint = safe_load(checkpoint_path)
    model.load_state_dict(
        checkpoint["model"],
        strict=True,
    )
    model.eval()

    @torch.no_grad()
    def collect_records(loader, description):
        records = []

        for images, masks, stems in tqdm(
            loader,
            desc=description,
            leave=False,
        ):
            images = images.to(
                DEVICE,
                non_blocking=True,
            )

            with torch.amp.autocast(
                "cuda",
                enabled=DEVICE.type == "cuda",
            ):
                probability = torch.sigmoid(
                    model(images)
                )

            records.append(
                {
                    "stem": stems[0],
                    "mask": masks[0, 0]
                    .float()
                    .numpy()
                    .astype(np.float32),
                    "probability": probability[0, 0]
                    .float()
                    .cpu()
                    .numpy()
                    .astype(np.float32),
                }
            )

        return records

    def threshold_search(records):
        rows = []

        for threshold in FINAL_THRESHOLDS:
            total = [0, 0, 0, 0]

            for record in records:
                total = add_values(
                    total,
                    confusion_numpy(
                        record["probability"],
                        record["mask"],
                        threshold,
                    ),
                )

            row = metrics(total)
            row["threshold"] = float(threshold)
            rows.append(row)

        best = max(
            rows,
            key=lambda item: item["dice"],
        )

        return best, rows

    def evaluate(records, threshold):
        total = [0, 0, 0, 0]
        per_image = []

        for record in records:
            values = confusion_numpy(
                record["probability"],
                record["mask"],
                threshold,
            )

            total = add_values(total, values)

            row = metrics(values)
            row.update(
                {
                    "stem": record["stem"],
                    "threshold": float(threshold),
                    "tp": values[0],
                    "fp": values[1],
                    "fn": values[2],
                    "tn": values[3],
                }
            )
            per_image.append(row)

        result = metrics(total)
        result.update(
            {
                "modality": modality,
                "seed": seed,
                "tta_mode": "no_tta",
                "threshold": float(threshold),
                "best_epoch": int(checkpoint["epoch"]),
                "best_validation_dice": float(
                    checkpoint["best_validation_dice"]
                ),
                "parameter_count": parameter_count,
                "tp": total[0],
                "fp": total[1],
                "fn": total[2],
                "tn": total[3],
            }
        )

        return result, per_image

    # Validation threshold search only.
    val_records = collect_records(
        val_loader,
        f"{modality} seed {seed} fine threshold validation",
    )

    val_best, threshold_rows = threshold_search(
        val_records
    )

    selected_threshold = float(
        val_best["threshold"]
    )

    save_csv(
        run_dir / "validation_threshold_search.csv",
        threshold_rows,
    )

    # Test is evaluated only after checkpoint + threshold are fixed.
    test_records = collect_records(
        test_loader,
        f"{modality} seed {seed} final test",
    )

    test_result, per_image_rows = evaluate(
        test_records,
        selected_threshold,
    )

    # The exact fine-grid validation Dice is useful to report separately from
    # the coarse checkpoint-selection Dice.
    test_result["fine_validation_dice"] = float(
        val_best["dice"]
    )

    save_csv(
        run_dir / "final_test_metrics.csv",
        [test_result],
    )

    save_csv(
        run_dir / "per_image_test_metrics.csv",
        per_image_rows,
    )

    with open(
        run_dir / "configuration.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "dataset": str(DATASET_DIR),
                "split": {
                    "train": len(TRAIN_NAMES),
                    "validation": len(VAL_NAMES),
                    "test": len(TEST_NAMES),
                },
                "modality": modality,
                "channels": info["channels"],
                "seed": seed,
                "architecture": "ResidualSCSEUNet",
                "encoder_channels": ENCODER_CHANNELS,
                "parameter_count": parameter_count,
                "target_size_width_height": list(TARGET_SIZE),
                "batch_size": BATCH_SIZE,
                "gradient_accumulation": ACCUMULATION_STEPS,
                "effective_batch_size": BATCH_SIZE * ACCUMULATION_STEPS,
                "optimizer": "AdamW",
                "initial_learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "max_epochs": MAX_EPOCHS,
                "early_stopping_patience": PATIENCE,
                "scheduler": "ReduceLROnPlateau",
                "scheduler_factor": 0.5,
                "scheduler_patience": 4,
                "minimum_learning_rate": 1e-6,
                "loss": "0.35 BCE + 0.25 focal + 0.40 Dice",
                "augmentation": (
                    "horizontal/vertical flips + RGB gain/bias"
                    if modality == "RGB"
                    else "horizontal/vertical flips; no MS radiometric perturbation"
                ),
                "mean": info["mean"].tolist(),
                "std": info["std"].tolist(),
                "coarse_training_thresholds": TRAIN_THRESHOLDS,
                "fine_validation_thresholds": FINAL_THRESHOLDS,
                "test_time_augmentation": False,
                "best_epoch": int(checkpoint["epoch"]),
                "best_coarse_validation_dice": float(
                    checkpoint["best_validation_dice"]
                ),
                "fine_validation_dice": float(val_best["dice"]),
                "selected_threshold": selected_threshold,
                "test_metrics": test_result,
            },
            f,
            indent=2,
        )

    print("\n" + "-" * 92)
    print(
        f"{modality}-ONLY | SEED {seed} | FINAL TEST"
    )
    print("-" * 92)
    print("Best epoch          :", checkpoint["epoch"])
    print(
        "Best validation Dice :",
        f"{val_best['dice']:.4f}",
    )
    print(
        "Threshold           :",
        f"{selected_threshold:.2f}",
    )
    print(
        "Dice                :",
        f"{test_result['dice']:.4f}",
    )
    print(
        "IoU                 :",
        f"{test_result['iou']:.4f}",
    )
    print(
        "Precision           :",
        f"{test_result['precision']:.4f}",
    )
    print(
        "Recall              :",
        f"{test_result['recall']:.4f}",
    )
    print(
        "Accuracy            :",
        f"{test_result['accuracy']:.4f}",
    )

    # Small per-run ZIP; no probability maps are saved.
    zip_path = shutil.make_archive(
        str(
            WORK_DIR
            / f"gd_{modality.lower()}_residual_scse_seed{seed}_results"
        ),
        "zip",
        str(run_dir),
    )

    print("Results ZIP          :", zip_path)

    del model
    del optimizer
    del scaler

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return test_result


# =============================================================================
# 8. RUN ONLY SEEDS 123 AND 2026
# =============================================================================

all_results = []

for seed in RUN_SEEDS:
    for modality in MODALITIES:
        result = run_experiment(
            seed,
            modality,
        )
        all_results.append(result)


# =============================================================================
# 9. SAVE ONLY THE FOUR NEW RESULTS — NO MEAN/SD
# =============================================================================

all_results = sorted(
    all_results,
    key=lambda row: (
        int(row["seed"]),
        str(row["modality"]),
    ),
)

summary_csv = (
    MASTER_OUT_DIR
    / "gd_rgb_ms_seed123_2026_results.csv"
)

save_csv(
    summary_csv,
    all_results,
)

print("\n" + "=" * 100)
print("FINAL SUMMARY — NEW RUNS ONLY")
print("=" * 100)

for row in all_results:
    print(
        f"Seed {int(row['seed'])} | "
        f"{row['modality']:3s} | "
        f"Dice={row['dice']:.4f} | "
        f"IoU={row['iou']:.4f} | "
        f"P={row['precision']:.4f} | "
        f"R={row['recall']:.4f} | "
        f"ValDice={row['fine_validation_dice']:.4f} | "
        f"th={row['threshold']:.2f} | "
        f"epoch={int(row['best_epoch'])}"
    )

combined_zip = shutil.make_archive(
    str(
        WORK_DIR
        / "gd_rgb_ms_residual_scse_seeds123_2026_complete"
    ),
    "zip",
    str(MASTER_OUT_DIR),
)

print("\nSummary CSV :", summary_csv)
print("Combined ZIP:", combined_zip)
print("\nSeed 42 was NOT rerun.")
print("Done.")
